# Stoichiometric Model Testing

This notebook tests the functionality of the stoichiometric reaction models:
- `AnabolicReaction`: Growth reaction stoichiometry
- `CatabolicReaction`: Maintenance reaction stoichiometry
- `StoichiometricModel`: Combined model for overall metabolism

**Suite role:** chemistry-first and stoichiometry-first validation with cross-mode comparison for presentation.

**Test Coverage:**
1. Loading reaction parameters from YAML files
2. Creating stoichiometric model
3. Calculating overall reactions at different growth rates
4. Calculating stoichiometric yields
5. Visualizing results
6. **Yield-based variable catabolism** *(new)*
   - Loading `cata_yield_based.yaml` (only `Y_Lac_Glc` and `Y_Gln_Glc` required)
   - Verifying atom-balance derived stoichiometry and `delta_Hf_catabolic`
   - Runtime update via `set_catabolic_yields()` including cascade re-computation
   - Unphysical yield guard (`O2 < 0` raises `ValueError`)
   - `Y_Lac_Glc` sweep showing sensitivity of stoichiometry and thermodynamics
   - Cross-model comparison table (explicit vs yield-based)
7. Three-mode comparison (`CONSTANT`, `DYNAMIC`, `EXCESS_COUPLED`) for q-rates and stoichiometric yields

**Date:** 2026-02-11 | **Updated:** 2026-03-13  
**Author:** Ben Tumulero

## 1. Import Libraries

In [29]:
import sys
from pathlib import Path

# Add src to path
repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Import animakinetica modules
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.stoichiometry import StoichiometricModel, StoichiometryMode

print("✓ Imports successful")

✓ Imports successful


## 3. Load Anabolic and Catabolic Reactions

In [16]:
# Load anabolic reaction
params = SimulationParameters.load_parameters(["base_parameters", "cell_GFI", "anab_GFI", "cata_GFI"])
stoich_model = StoichiometricModel(params)
# Display anabolic reaction coefficients
print("\n" + "="*60)
print("ANABOLIC REACTION (Growth)")
print("="*60)
anabolic_dict = stoich_model.anabolic_dict
print("\nStoichiometric coefficients (per mol DCM produced):")
for species, coeff in anabolic_dict.items():
    sign = "→" if coeff > 0 else "←"
    print(f"  {species:5s} {sign} {coeff:+.4f}")

lumped_AA_coeff = stoich_model.AA_anabolic_coeff
print(f"\nLumped amino acid coefficient (per mol DCM): {lumped_AA_coeff:.4f}")

print(f"\nThermodynamics:")
print(f"  ΔHf = {params.anabolic_reaction.delta_Hf.value:.2f} {params.anabolic_reaction.delta_Hf.unit}")
print(f"  ΔGf = {params.anabolic_reaction.delta_Gf.value:.2f} {params.anabolic_reaction.delta_Gf.unit}")

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\cell_properties\cell_GFI.yaml
  [3/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 4 file(s).

ANABOLIC REACTION (Growth)

Stoichiometric coefficients (per mol DCM produced):
  Glc   ← -0.0770
  Arg   ← -0.0059
  Cys   ← -0.0035
  Gln   ← -0.0508
  His   ← -0.0026
  Ile   ← -0.0058
  Lys   ← -0.0088
  Met   ← -0.0021
  Phe   ← -0.0044
  Thr   ← -0.0075
  Trp   ← -0.0014
  Tyr   ← -0.0044
  Val   ← -0

In [17]:
# Display catabolic reaction coefficients
print("\n" + "="*60)
print("CATABOLIC REACTION (Maintenance)")
print("="*60)
catabolic_dict = stoich_model.catabolic_dict
print("\nStoichiometric coefficients:")
for species, coeff in catabolic_dict.items():
    sign = "→" if coeff > 0 else "←"
    print(f"  {species:5s} {sign} {coeff:+.4f}")

print(f"\nThermodynamics:")
print(f"  ΔHf = {params.catabolic_reaction.delta_Hf.value:.2f} {params.catabolic_reaction.delta_Hf.unit}")
print(f"  ΔGf = {params.catabolic_reaction.delta_Gf.value:.2f} {params.catabolic_reaction.delta_Gf.unit}")


CATABOLIC REACTION (Maintenance)

Stoichiometric coefficients:
  Glc   ← -1.0000
  Gln   ← -0.1371
  O2    ← -4.1214
  NH3   → +0.2742
  Lac   → +0.8319
  CO2   → +4.1899
  H2O   → +3.7786

Thermodynamics:
  ΔHf = -1986.26 kJ/mol
  ΔGf = -2029.88 kJ/mol


## 4. Test species list

In [18]:
print("✓ Stoichiometric model created successfully")
print(f"\n  Unified species list ({len(stoich_model.species_list)} species):")
print(f"    {', '.join(stoich_model.species_list)}")
print(f"\n  Catabolic extent: {stoich_model.cat_extent:.6e} mol h⁻¹ Cmol⁻¹")
print(f"\n  Cell properties:")
print(f"    Wet cell mass: {params.cell_properties.wet_cell_mass * 1e12:.2f} pg/cell")
print(f"    Dry cell mass: {params.cell_properties.dry_cell_mass * 1e12:.2f} pg/cell")
print(f"    M_biomass: {params.cell_properties.M_biomass:.2f} gDCW/Cmol")

✓ Stoichiometric model created successfully

  Unified species list (27 species):
    Ala, Arg, Asn, Asp, CO2, Cys, DCM, Glc, Gln, Glu, Gly, H2O, His, Ile, Lac, Leu, Lys, Met, NH3, O2, Phe, Pro, Ser, Thr, Trp, Tyr, Val

  Catabolic extent: 6.947582e-03 mol h⁻¹ Cmol⁻¹

  Cell properties:
    Wet cell mass: 1000.00 pg/cell
    Dry cell mass: 150.00 pg/cell
    M_biomass: 21.85 gDCW/Cmol


## 5. Calculate Overall Reactions at Different Growth Rates

In [19]:
# Define range of growth rates to test
mu_values = np.array([0.01, 0.02, 0.029, 0.03, 0.03851, 0.04, 0.05])  # 1/h

# Calculate overall reactions
overall_reactions = {}
for mu in mu_values:
    stoich_model._update_overall_reaction_array(mu)
    overall_reactions[mu] = stoich_model.overall_reaction_dict

# Display results for each growth rate
print("\n" + "="*80)
print("OVERALL REACTIONS AT DIFFERENT GROWTH RATES")
print("="*80)

for mu, overall in overall_reactions.items():
    print(f"\nμ = {mu:.3f} h⁻¹")
    print("-" * 40)
    for species, coeff in overall.items():
        if abs(coeff) > 1e-10:  # Only show non-zero coefficients
            sign = "produces" if coeff > 0 else "consumes"
            print(f"  {species:5s}: {sign:8s} {abs(coeff):8.4f} mol")


OVERALL REACTIONS AT DIFFERENT GROWTH RATES

μ = 0.010 h⁻¹
----------------------------------------
  Ala  : consumes   0.0113 mol
  Arg  : consumes   0.0059 mol
  Asn  : consumes   0.0055 mol
  Asp  : consumes   0.0069 mol
  CO2  : produces   2.9794 mol
  Cys  : consumes   0.0035 mol
  DCM  : produces   1.0000 mol
  Glc  : consumes   0.7718 mol
  Gln  : consumes   0.1461 mol
  Glu  : produces   0.0381 mol
  Gly  : consumes   0.0094 mol
  H2O  : produces   2.8923 mol
  His  : consumes   0.0026 mol
  Ile  : consumes   0.0058 mol
  Lac  : produces   0.5780 mol
  Leu  : consumes   0.0094 mol
  Lys  : consumes   0.0088 mol
  Met  : consumes   0.0021 mol
  NH3  : produces   0.1905 mol
  O2   : consumes   2.8634 mol
  Phe  : consumes   0.0044 mol
  Pro  : consumes   0.0058 mol
  Ser  : consumes   0.0089 mol
  Thr  : consumes   0.0075 mol
  Trp  : consumes   0.0014 mol
  Tyr  : consumes   0.0044 mol
  Val  : consumes   0.0087 mol

μ = 0.020 h⁻¹
----------------------------------------
  Ala 

## 6. Calculate Stoichiometric Yields

## GFI Cell

In [20]:
# Yields are obtained in a Numpy array of yield coefficients [mol/mol] in the following order:
#     [0] Y_Glc/X: Glucose consumed per biomass produced [mol_Glc / mol_DCM]
#     [1] Y_Lac/Glc: Lactate produced per glucose consumed [mol_Lac / mol_Glc]
#     [2] Y_Gln/Glc: Glutamine consumed per glucose consumed [mol_Gln / mol_Glc]
#     [3] Y_O2/Glc: Oxygen consumed per glucose consumed [mol_O2 / mol_Glc]
#     [4] Y_CO2/Glc: CO2 produced per glucose consumed [mol_CO2 / mol_Glc]
#     [5] Y_AA/Glc: Amino acids consumed per glucose consumed [mol_AA / mol_Glc]
#     [6] Y_Amm/Gln: Ammonia produced per glutamine consumed [mol_NH3 / mol_Gln]
#     [7] Y_H2O/X: Metabolic water produced per biomass produced [mol_H2O / mol_DCM]


# Define yield column names
yield_columns = [
    'Y_Glc/X', 'Y_Lac/Glc', 'Y_Gln/Glc', 'Y_O2/Glc', 'Y_CO2/Glc',
    'Y_AA/Glc', 'Y_Amm/Gln', 'Y_H2O/X',
]

# Use DYNAMIC mode so yields are recalculated at each mu -> gradient variation is meaningful
stoich_model_dynamic = StoichiometricModel(params, mode=StoichiometryMode.DYNAMIC)

# Calculate yields for each growth rate
yields_data = {}
for mu in mu_values:
    yields = stoich_model_dynamic.calculate_yields(mu)
    yields_data[mu] = yields

# Convert to DataFrame for easier visualization
yields_df = pd.DataFrame(yields_data).T
yields_df.columns = yield_columns
yields_df.index.name = 'μ [1/h]'

print("\n" + "="*80)
print("STOICHIOMETRIC YIELDS")
print("="*80)
print("\nYields [mol/mol]:")
print(yields_df.to_string(float_format=lambda x: f"{x:.4f}"))

# Display as styled table (axis=0: per-column gradient; variation visible across μ values)
display(yields_df.style.format("{:.4f}").background_gradient(cmap='viridis', axis=0))


STOICHIOMETRIC YIELDS

Yields [mol/mol]:
         Y_Glc/X  Y_Lac/Glc  Y_Gln/Glc  Y_O2/Glc  Y_CO2/Glc  Y_AA/Glc  Y_Amm/Gln  Y_H2O/X
μ [1/h]                                                                                  
0.01000   0.7718     0.7488     0.1892    3.7099     3.8604    0.0964     1.3044   2.8923
0.02000   0.4244     0.6809     0.2319    3.3732     3.5906    0.1753     0.9678   1.5797
0.02900   0.3166     0.6294     0.2642    3.1184     3.3865    0.2350     0.7854   1.1723
0.03000   0.3086     0.6242     0.2675    3.0925     3.3658    0.2410     0.7693   1.1421
0.03851   0.2575     0.5829     0.2934    2.8880     3.2019    0.2889     0.6550   0.9488
0.04000   0.2507     0.5763     0.2976    2.8549     3.1754    0.2967     0.6384   0.9234
0.05000   0.2160     0.5351     0.3234    2.6512     3.0123    0.3444     0.5455   0.7921


,Y_Glc/X,Y_Lac/Glc,Y_Gln/Glc,Y_O2/Glc,Y_CO2/Glc,Y_AA/Glc,Y_Amm/Gln,Y_H2O/X
μ [1/h],,,,,,,,
0.010000,0.7718,0.7488,0.1892,3.7099,3.8604,0.0964,1.3044,2.8923
0.020000,0.4244,0.6809,0.2319,3.3732,3.5906,0.1753,0.9678,1.5797
0.029000,0.3166,0.6294,0.2642,3.1184,3.3865,0.2350,0.7854,1.1723
0.030000,0.3086,0.6242,0.2675,3.0925,3.3658,0.2410,0.7693,1.1421
0.038510,0.2575,0.5829,0.2934,2.8880,3.2019,0.2889,0.6550,0.9488
0.040000,0.2507,0.5763,0.2976,2.8549,3.1754,0.2967,0.6384,0.9234
0.050000,0.2160,0.5351,0.3234,2.6512,3.0123,0.3444,0.5455,0.7921


## Enhanced Humbird Cell

In [21]:
params_enhanced = SimulationParameters.load_parameters(["base_parameters", "cell_Humbird", "anab_Humbird_cell", "cata_Humbird_enhanced"])
stoich_model_enhanced = StoichiometricModel(params_enhanced)
stoich_model_enhanced_dynamic = StoichiometricModel(params_enhanced, mode=StoichiometryMode.DYNAMIC)

# Calculate yields for each growth rate
yields_data = {}
for mu in mu_values:
    yields = stoich_model_enhanced_dynamic.calculate_yields(mu)
    yields_data[mu] = yields

# Convert to DataFrame for easier visualization
yields_df = pd.DataFrame(yields_data).T
yields_df.columns = yield_columns
yields_df.index.name = 'μ [1/h]'

print("\n" + "="*80)
print("STOICHIOMETRIC YIELDS")
print("="*80)
print("\nYields [mol/mol]:")
print(yields_df.to_string(float_format=lambda x: f"{x:.4f}"))

# Display as styled table
display(yields_df.style.format("{:.4f}").background_gradient(cmap='viridis', axis=0))

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\cell_properties\cell_Humbird.yaml
  [3/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\stoichiometry\anab_Humbird_cell.yaml
  [4/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\stoichiometry\cata_Humbird_enhanced.yaml
Successfully loaded parameters from 4 file(s).

STOICHIOMETRIC YIELDS

Yields [mol/mol]:
         Y_Glc/X  Y_Lac/Glc  Y_Gln/Glc  Y_O2/Glc  Y_CO2/Glc  Y_AA/Glc  Y_Amm/Gln  Y_H2O/X
μ [1/h]                                                                                  
0.01000   0.3017     0.3921     0.0851    3.6174     3.8794  

,Y_Glc/X,Y_Lac/Glc,Y_Gln/Glc,Y_O2/Glc,Y_CO2/Glc,Y_AA/Glc,Y_Amm/Gln,Y_H2O/X
μ [1/h],,,,,,,,
0.010000,0.3017,0.3921,0.0851,3.6174,3.8794,0.4038,0.4608,1.3170
0.020000,0.1834,0.3225,0.1239,2.9754,3.3985,0.6642,0.2604,0.7787
0.029000,0.1467,0.2781,0.1486,2.5656,3.0914,0.8305,0.1872,0.6117
0.030000,0.1440,0.2739,0.1509,2.5270,3.0625,0.8462,0.1815,0.5993
0.038510,0.1265,0.2428,0.1683,2.2397,2.8472,0.9627,0.1443,0.5200
0.040000,0.1242,0.2380,0.1709,2.1960,2.8145,0.9805,0.1393,0.5096
0.050000,0.1124,0.2105,0.1863,1.9417,2.6240,1.0837,0.1130,0.4557


## Yield-based variable catabolism

Tests the atom-balance path: `cata_yield_based.yaml` provides only `Y_Lac_Glc` and `Y_Gln_Glc`;
all other stoichiometric coefficients and `delta_Hf_catabolic` are derived automatically.

In [22]:
# --- Load yield-based catabolic configuration ---
params_yb = SimulationParameters.load_parameters([
    "base_parameters",
    "cell_Humbird",
    "anab_Humbird_cell",
    "cata_Humbird_wildtype_yield_based",   # provides only Y_Lac_Glc=0.50, Y_Gln_Glc=0.025
])

stoich_model_yb = StoichiometricModel(params_yb)

# Print the two input yields from the YAML
cat_yb = params_yb.catabolic_reaction
print("Yield-based catabolic inputs:")
print(f"  Y_Lac_Glc = {cat_yb.Y_Lac_Glc.value:.4f}  {cat_yb.Y_Lac_Glc.unit}")
print(f"  Y_Gln_Glc = {cat_yb.Y_Gln_Glc.value:.4f}  {cat_yb.Y_Gln_Glc.unit}")

# Show the derived stoichiometric coefficients (result of atom balance)
print("\nDerived catabolic stoichiometry (atom balance, per mol Glc):")
for species, coeff in stoich_model_yb.catabolic_dict.items():
    sign = "→" if coeff > 0 else "←"
    print(f"  {species:5s} {sign} {coeff:+.4f}")

# Show derived cat_extent and delta_Hf
from animakinetica.models.stoichiometry import _CATABOLIC_SPECIES_DHF
dhf_derived = StoichiometricModel._calculate_dhf_from_stoich(stoich_model_yb.catabolic_dict)
print(f"\nDerived delta_Hf_catabolic = {dhf_derived:.4f} kJ/mol")
print(f"Catabolic extent           = {stoich_model_yb.cat_extent:.6e} mol h⁻¹ Cmol⁻¹")

# Verify against explicit-coefficient Humbird wildtype model (should be close for same reference yields)
print("\n--- Comparison: explicit Humbird wildtype ---")
for species, coeff in stoich_model.catabolic_dict.items():
    sign_w = "→" if coeff > 0 else "←"
    yb_coeff = stoich_model_yb.catabolic_dict.get(species, 0.0)
    print(f"  {species:5s}: wildtype={coeff:+.4f}  yield-based={yb_coeff:+.4f}")

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\cell_properties\cell_Humbird.yaml
  [3/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\stoichiometry\anab_Humbird_cell.yaml
  [4/4] Updating with: c:\Users\btumu\OneDrive - Delft University of Technology\2. Code\animakinetica\config\parameter_sets\stoichiometry\cata_Humbird_wildtype_yield_based.yaml
Successfully loaded parameters from 4 file(s).
Yield-based catabolic inputs:
  Y_Lac_Glc = 1.7700  mol/mol
  Y_Gln_Glc = 0.1300  mol/mol

Derived catabolic stoichiometry (atom balance, per mol Glc):
  Glc   ← -1.0000
  Gln   ← -0.1300
  O2    ← -1.2750
  NH3   → +0.2600
  Lac   → +1.7700
  CO2   → +1.3400
  H2O   → +

In [23]:
# --- Test set_catabolic_yields() runtime update ---
print("Testing set_catabolic_yields() runtime update")
print("="*60)

mu_test = params_yb.cell_properties.mu_max.value  # float, not Parameter

# Baseline yields at default values (0.50, 0.025)
yields_before = stoich_model_yb.calculate_yields(mu_test).copy()
cat_extent_before = stoich_model_yb.cat_extent

print(f"Before update  (Y_Lac_Glc=0.500, Y_Gln_Glc=0.025):")
print(f"  cat_extent      = {cat_extent_before:.6e} mol h⁻¹ Cmol⁻¹")
print(f"  delta_Hf_cat    = {StoichiometricModel._calculate_dhf_from_stoich(stoich_model_yb.catabolic_dict):.4f} kJ/mol")
print(f"  Y_Lac/Glc       = {yields_before[1]:.4f}")
print(f"  Y_Gln/Glc       = {yields_before[2]:.4f}")
print(f"  Y_O2/Glc        = {yields_before[3]:.4f}")

# Update to new yields
stoich_model_yb.set_catabolic_yields(Y_Lac_Glc=0.30, Y_Gln_Glc=0.05)

yields_after = stoich_model_yb.calculate_yields(mu_test).copy()
cat_extent_after = stoich_model_yb.cat_extent

print(f"\nAfter update   (Y_Lac_Glc=0.300, Y_Gln_Glc=0.050):")
print(f"  cat_extent      = {cat_extent_after:.6e} mol h⁻¹ Cmol⁻¹")
print(f"  delta_Hf_cat    = {StoichiometricModel._calculate_dhf_from_stoich(stoich_model_yb.catabolic_dict):.4f} kJ/mol")
print(f"  Y_Lac/Glc       = {yields_after[1]:.4f}")
print(f"  Y_Gln/Glc       = {yields_after[2]:.4f}")
print(f"  Y_O2/Glc        = {yields_after[3]:.4f}")

# Verify param objects were updated
print(f"\nParameter objects updated:")
print(f"  params_yb.catabolic_reaction.Y_Lac_Glc.value = {params_yb.catabolic_reaction.Y_Lac_Glc.value}")
print(f"  params_yb.catabolic_reaction.Y_Gln_Glc.value = {params_yb.catabolic_reaction.Y_Gln_Glc.value}")

# Reset to original values for downstream cells
stoich_model_yb.set_catabolic_yields(Y_Lac_Glc=0.50, Y_Gln_Glc=0.025)
print("\n✓ Reset to original values (Y_Lac_Glc=0.500, Y_Gln_Glc=0.025)")

# Test error handling for unphysical values
print("\nTesting unphysical yield guard...")
try:
    stoich_model_yb.set_catabolic_yields(Y_Lac_Glc=3.0, Y_Gln_Glc=0.0)  # Forces negative O2
    print("  ✗ No error raised (unexpected)")
except ValueError as e:
    print(f"  ✓ ValueError raised as expected: {e}")

Testing set_catabolic_yields() runtime update
Before update  (Y_Lac_Glc=0.500, Y_Gln_Glc=0.025):
  cat_extent      = 7.737662e-03 mol h⁻¹ Cmol⁻¹
  delta_Hf_cat    = -680.5012 kJ/mol
  Y_Lac/Glc       = 1.4240
  Y_Gln/Glc       = 0.1639
  Y_O2/Glc        = 1.0258

After update   (Y_Lac_Glc=0.300, Y_Gln_Glc=0.050):
  cat_extent      = 2.065364e-03 mol h⁻¹ Cmol⁻¹
  delta_Hf_cat    = -2549.4245 kJ/mol
  Y_Lac/Glc       = 0.1570
  Y_Gln/Glc       = 0.1708
  Y_O2/Glc        = 2.7876

Parameter objects updated:
  params_yb.catabolic_reaction.Y_Lac_Glc.value = 0.3
  params_yb.catabolic_reaction.Y_Gln_Glc.value = 0.05

✓ Reset to original values (Y_Lac_Glc=0.500, Y_Gln_Glc=0.025)

Testing unphysical yield guard...
  ✓ ValueError raised as expected: Unphysical catabolic stoichiometry: O2 coefficient = 3.0000 > 0 (O2 would be produced rather than consumed) for Y_Lac_Glc=3.0, Y_Gln_Glc=0.0.  Reduce Y_Lac_Glc or Y_Gln_Glc.


In [24]:
# --- Sweep Y_Lac_Glc; inspect resulting stoichiometry and overall yields ---
Y_Lac_sweep = np.linspace(0.1, 1.8, 9)
Y_Gln_fixed = 0.025
mu_max_val   = params_yb.cell_properties.mu_max.value  # float

stoich_display_cols = ['Glc demand', 'Gln demand', 'O2 demand', 'NH3', 'Lac', 'CO2', 'H2O']

sweep_yields  = []
sweep_stoich  = []
sweep_dhf     = []
sweep_cat_ext = []
valid_Y_Lac   = []

for Y_Lac in Y_Lac_sweep:
    # test = StoichiometricModel._atom_balance_catabolic(Y_Lac, Y_Gln_fixed)
    # # O2 is stored as a negative stoichiometric coefficient (substrate).
    # # test['O2'] > 0 means O2 would be produced — unphysical, skip.
    # if test['O2'] > 0:
    #     continue
    valid_Y_Lac.append(Y_Lac)
    sweep_yields.append(stoich_model_yb.calculate_yields(mu_max_val, (Y_Lac, Y_Gln_fixed)).copy())
    # Store substrates as positive demands for readability
    cd = stoich_model_yb.catabolic_dict
    sweep_stoich.append([abs(cd['Glc']), abs(cd['Gln']), abs(cd['O2']),
                         cd['NH3'], cd['Lac'], cd['CO2'], cd['H2O']])
    dhf = StoichiometricModel._calculate_dhf_from_stoich(stoich_model_yb.catabolic_dict)
    sweep_dhf.append(dhf)
    sweep_cat_ext.append(stoich_model_yb.cat_extent)

# Reset
stoich_model_yb.set_catabolic_yields(0.50, 0.025)

df_sweep_yields = pd.DataFrame(sweep_yields, index=valid_Y_Lac, columns=yield_columns)
df_sweep_yields.index.name = 'Y_Lac_Glc'

df_sweep_stoich = pd.DataFrame(sweep_stoich, index=valid_Y_Lac, columns=stoich_display_cols)
df_sweep_stoich['ΔHf [kJ/mol]']             = sweep_dhf
df_sweep_stoich['cat_extent [mol/h/Cmol]'] = sweep_cat_ext
df_sweep_stoich.index.name = 'Y_Lac_Glc'

print(f"Valid Y_Lac_Glc values: {valid_Y_Lac}")
print("\nDerived catabolic stoichiometry & thermodynamics over Y_Lac_Glc sweep (Y_Gln_Glc=0.025):")
print("  (substrate columns shown as demands — positive values)")
display(df_sweep_stoich.style.format("{:.4f}").background_gradient(cmap='RdYlGn', axis=0))

print("\nOverall yields at μ_max over Y_Lac_Glc sweep:")
display(df_sweep_yields.style.format("{:.4f}").background_gradient(cmap='RdYlGn', axis=0))

Valid Y_Lac_Glc values: [np.float64(0.1), np.float64(0.3125), np.float64(0.525), np.float64(0.7374999999999999), np.float64(0.95), np.float64(1.1625), np.float64(1.375), np.float64(1.5875000000000001), np.float64(1.8)]

Derived catabolic stoichiometry & thermodynamics over Y_Lac_Glc sweep (Y_Gln_Glc=0.025):
  (substrate columns shown as demands — positive values)


,Glc demand,Gln demand,O2 demand,NH3,Lac,CO2,H2O,ΔHf [kJ/mol],cat_extent [mol/h/Cmol]
Y_Lac_Glc,,,,,,,,,
0.100000,1.0000,0.0250,5.8125,0.0500,0.1000,5.8250,5.7500,-2776.2648,0.0019
0.312500,1.0000,0.0250,5.1750,0.0500,0.3125,5.1875,5.1125,-2483.6204,0.0021
0.525000,1.0000,0.0250,4.5375,0.0500,0.5250,4.5500,4.4750,-2190.9760,0.0024
0.737500,1.0000,0.0250,3.9000,0.0500,0.7375,3.9125,3.8375,-1898.3316,0.0028
0.950000,1.0000,0.0250,3.2625,0.0500,0.9500,3.2750,3.2000,-1605.6872,0.0033
1.162500,1.0000,0.0250,2.6250,0.0500,1.1625,2.6375,2.5625,-1313.0429,0.0040
1.375000,1.0000,0.0250,1.9875,0.0500,1.3750,2.0000,1.9250,-1020.3985,0.0052
1.587500,1.0000,0.0250,1.3500,0.0500,1.5875,1.3625,1.2875,-727.7541,0.0072
1.800000,1.0000,0.0250,0.7125,0.0500,1.8000,0.7250,0.6500,-435.1097,0.0121



Overall yields at μ_max over Y_Lac_Glc sweep:


,Y_Glc/X,Y_Lac/Glc,Y_Gln/Glc,Y_O2/Glc,Y_CO2/Glc,Y_AA/Glc,Y_Amm/Gln,Y_H2O/X
Y_Lac_Glc,,,,,,,,
0.100000,0.1308,0.0502,0.1636,2.9191,3.5074,0.9316,0.1534,0.6180
0.312500,0.1385,0.1656,0.1559,2.7428,3.2990,0.8796,0.1700,0.6157
0.525000,0.1483,0.2946,0.1472,2.5459,3.0661,0.8214,0.1905,0.6128
0.737500,0.1611,0.4396,0.1375,2.3245,2.8043,0.7560,0.2167,0.6090
0.950000,0.1786,0.6038,0.1265,2.0737,2.5077,0.6819,0.2513,0.6038
1.162500,0.2039,0.7915,0.1139,1.7872,2.1689,0.5973,0.2989,0.5962
1.375000,0.2438,1.0078,0.0994,1.4568,1.7782,0.4997,0.3688,0.5844
1.587500,0.3156,1.2601,0.0824,1.0716,1.3226,0.3860,0.4814,0.5630
1.800000,0.4841,1.5580,0.0624,0.6167,0.7847,0.2516,0.6930,0.5128


In [25]:
# --- Visualise: Y_Lac_Glc sweep — impact on key catabolic stoichiometry ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'O₂ demand  [mol / mol Glc]',
        'CO₂ produced [mol / mol Glc]',
        'ΔHf_catabolic [kJ / mol Glc]',
        'Catabolic extent [mol h⁻¹ Cmol⁻¹]',
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.12,
)

x = df_sweep_stoich.index.values

fig.add_trace(go.Scatter(x=x, y=df_sweep_stoich['O2 demand'], mode='lines+markers', name='O2 demand',    line=dict(color='royalblue',  width=2), marker=dict(size=7)), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=df_sweep_stoich['CO2'],       mode='lines+markers', name='CO2 produced', line=dict(color='firebrick',  width=2), marker=dict(size=7)), row=1, col=2)
fig.add_trace(go.Scatter(x=x, y=df_sweep_stoich['ΔHf [kJ/mol]'],           mode='lines+markers', name='ΔHf_cat',    line=dict(color='darkorange', width=2), marker=dict(size=7)), row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=df_sweep_stoich['cat_extent [mol/h/Cmol]'], mode='lines+markers', name='cat_extent', line=dict(color='seagreen',   width=2), marker=dict(size=7)), row=2, col=2)

fig.update_xaxes(title_text='Y_Lac_Glc [mol/mol]')
fig.update_layout(
    title=dict(text='Yield-based catabolism: sensitivity to Y_Lac_Glc (Y_Gln_Glc = 0.025)', x=0.5, xanchor='center'),
    height=700, showlegend=False,
)
fig.show()

# --- Visualise: overall yields at mu_max for all four models ---
mu_ref = params.cell_properties.mu_max.value  # float


models = {
    'GFI (explicit)'    : stoich_model,
    'Humbird Enhanced (explicit)'    : stoich_model_enhanced,
    #'GFI-Humbird Wildtype (explicit)' : stoich_model_GFIHumbird,
    'Humbird Enhanced Yield-based (0.5/0.025)': stoich_model_yb,
}

comparison_rows = []
for label, m in models.items():
    y = m.calculate_yields(mu_ref)
    comparison_rows.append([label] + list(y))

df_compare = pd.DataFrame(comparison_rows, columns=['Model'] + yield_columns).set_index('Model')
print(f"\nOverall yields at μ_max = {mu_ref:.4f} h⁻¹ — all models:")
# axis=None: single colour scale across entire table -> differences between models visible
display(df_compare.style.format("{:.4f}").background_gradient(cmap='YlOrRd', axis=None))


Overall yields at μ_max = 0.0385 h⁻¹ — all models:


,Y_Glc/X,Y_Lac/Glc,Y_Gln/Glc,Y_O2/Glc,Y_CO2/Glc,Y_AA/Glc,Y_Amm/Gln,Y_H2O/X
Model,,,,,,,,
GFI (explicit),0.2575,0.5829,0.2934,2.8880,3.2020,0.2889,0.6550,0.9488
Humbird Enhanced (explicit),0.1470,0.2786,0.1483,2.5703,3.0949,0.8286,0.1879,0.6132
Humbird Enhanced Yield-based (0.5/0.025),0.1470,0.2786,0.1483,2.5703,3.0949,0.8286,0.1879,0.6132


## 7. Visualize Overall Reaction Coefficients vs Growth Rate

In [26]:
# Prepare data for plotting
species_to_plot = ['Glc', 'Gln', 'O2', 'Lac', 'NH3', 'CO2', 'DCM']

# Create subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Substrate Consumption (negative values)', 'Product Formation (positive values)'),
    vertical_spacing=0.12
)

# Extract data for each species
for species in species_to_plot:
    coeffs = [overall_reactions[mu].get(species, 0) for mu in mu_values]
    
    # Determine if substrate or product
    is_substrate = all(c <= 0 for c in coeffs if abs(c) > 1e-10)
    row = 1 if is_substrate else 2
    
    fig.add_trace(
        go.Scatter(
            x=mu_values,
            y=coeffs,
            mode='lines+markers',
            name=species,
            line=dict(width=2),
            marker=dict(size=8)
        ),
        row=row, col=1
    )

# Update layout
fig.update_xaxes(title_text="Growth Rate μ [1/h]", row=2, col=1)
fig.update_yaxes(title_text="Stoichiometric Coefficient [mol]", row=1, col=1)
fig.update_yaxes(title_text="Stoichiometric Coefficient [mol]", row=2, col=1)

fig.update_layout(
    title=dict(
        text="Overall Reaction Stoichiometry vs Growth Rate",
        x=0.5,
        xanchor='center'
    ),
    height=800,
    showlegend=True,
    hovermode='x unified'
)

fig.show()

## 8. Visualize Yield Coefficients vs Growth Rate

In [27]:
# Create figure for yields
fig = go.Figure()

for yield_name in yields_df.columns:
    fig.add_trace(
        go.Scatter(
            x=yields_df.index,
            y=yields_df[yield_name],
            mode='lines+markers',
            name=yield_name,
            line=dict(width=2),
            marker=dict(size=8)
        )
    )

fig.update_layout(
    title=dict(
        text="Stoichiometric Yields vs Growth Rate",
        x=0.5,
        xanchor='center'
    ),
    xaxis_title="Growth Rate μ [1/h]",
    yaxis_title="Yield Coefficient [mol/mol]",
    height=600,
    showlegend=True,
    hovermode='x unified'
)

fig.show()

## 10. Mass Balance Check

In [28]:
# Verify carbon balance for overall reaction
# C-mol content of each species
carbon_content = {
    'Glc': 6,    # Glucose: C6H12O6
    'Gln': 5,    # Glutamine: C5H10N2O3
    'Lac': 3,    # Lactate: C3H5O3
    'CO2': 1,    # Carbon dioxide
    'NH3': 0,    # Ammonia
    'O2': 0,     # Oxygen
    'H2O': 0,    # Water
    'DCM': 1,    # Dry cell mass (normalized to 1 C-mol)
    # Amino acids
    'Ala': 3, 'Arg': 6, 'Asn': 4, 'Asp': 4, 'Cys': 3,
    'Glu': 5, 'Gly': 2, 'His': 6, 'Ile': 6, 'Leu': 6,
    'Lys': 6, 'Met': 5, 'Phe': 9, 'Pro': 5, 'Ser': 3,
    'Thr': 4, 'Trp': 11, 'Tyr': 9, 'Val': 5
}

print("\n" + "="*60)
print("CARBON MASS BALANCE CHECK")
print("="*60)

for mu in [0.01, 0.03, 0.05]:
    stoich_model._update_overall_reaction_array(mu)
    stoich_model_enhanced._update_overall_reaction_array(mu)

    overall = stoich_model.overall_reaction_dict
    overall_enhanced = stoich_model_enhanced.overall_reaction_dict
    
    print(f"\nμ = {mu:.3f} h⁻¹:")
    print("=" * 80)
    
    # Check all three models
    for model_name, overall_rxn in [
        ("GFI", overall),
        ("Humbird Enhanced", overall_enhanced),
    ]:
        # Calculate carbon balance
        carbon_in = 0
        carbon_out = 0
        
        for species, coeff in overall_rxn.items():
            if species in carbon_content:
                c_mol = carbon_content[species] * coeff
                if coeff < 0:  # Consumption (negative)
                    carbon_in += abs(c_mol)
                else:  # Production (positive)
                    carbon_out += c_mol
        
        balance = carbon_out - carbon_in
        rel_error = abs(balance) / carbon_in * 100 if carbon_in > 0 else 0
        
        print(f"\n  {model_name}:")
        print(f"    C-mol in:  {carbon_in:.6f}")
        print(f"    C-mol out: {carbon_out:.6f}")
        print(f"    Balance:   {balance:+.6f} ({rel_error:.4f}%)")
        
        if abs(rel_error) < 0.1:
            print(f"    ✓ Carbon balance closed!")
        else:
            print(f"    ✗ Carbon balance error!")


CARBON MASS BALANCE CHECK

μ = 0.010 h⁻¹:

  GFI:
    C-mol in:  5.903898
    C-mol out: 5.903862
    Balance:   -0.000036 (0.0006%)
    ✓ Carbon balance closed!

  Humbird Enhanced:
    C-mol in:  2.552336
    C-mol out: 2.552336
    Balance:   -0.000000 (0.0000%)
    ✓ Carbon balance closed!

μ = 0.030 h⁻¹:

  GFI:
    C-mol in:  2.807316
    C-mol out: 2.807280
    Balance:   -0.000036 (0.0013%)
    ✓ Carbon balance closed!

  Humbird Enhanced:
    C-mol in:  1.586184
    C-mol out: 1.586184
    Balance:   -0.000000 (0.0000%)
    ✓ Carbon balance closed!

μ = 0.050 h⁻¹:

  GFI:
    C-mol in:  2.187999
    C-mol out: 2.187963
    Balance:   -0.000036 (0.0017%)
    ✓ Carbon balance closed!

  Humbird Enhanced:
    C-mol in:  1.392954
    C-mol out: 1.392954
    Balance:   -0.000000 (0.0000%)
    ✓ Carbon balance closed!
